In [1]:
#!pip uninstall -y autogluon numpy scikit-learn
!pip install numpy==1.26.4
!pip install scikit-learn==1.3.2
!pip install autogluon.tabular[all]==1.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 86.5 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 kB 5.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.1/154.1 kB 8.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with ot

In [2]:
import pandas as pd
import os
import re
import string
import math
import numpy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
# Set of English stopwords for later calculations
stopwords_set = set(stopwords.words('english'))
MODEL = 'AutoML'#Regression

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
def load_data(data_dir, train_csv_path):
    """Load and process training data with better error handling"""
    train_df = pd.read_csv(train_csv_path)
    data = []

    for _, row in train_df.iterrows():
        folder_id = str(row.iloc[0])
        real_text_id = str(row.iloc[1])


        folder_name = f"article_{folder_id.zfill(4)}"
        folder_path = os.path.join(data_dir, folder_name)

        for file_id in ["1", "2"]:
            file_path = os.path.join(folder_path, f"file_{file_id}.txt")

            try:
                with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                    text = f.read().strip()
                if not text:
                    continue

                label = 1 if file_id == real_text_id else 0
                data.append({'text': text, 'real': label, 'folder': folder_id})

            except (FileNotFoundError, UnicodeDecodeError) as e:
                print(f"⚠️ Error loading {file_path}: {str(e)}")
                continue

    df = pd.DataFrame(data)
    return df.dropna(subset=['text'])

In [4]:
class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()
        self.punct_table = str.maketrans('', '', string.punctuation)

    def preprocess(self, text):
        text = text.lower()

        text = text.translate(self.punct_table)

        tokens = [self.lemmatizer.lemmatize(word)
                 for word in text.split()
                 if word not in self.stop_words]

        return ' '.join(tokens)

In [5]:
def entropy(text):
    """Calculate the Shannon entropy of the text."""
    if len(text) == 0:
        return 0
    probs = [v / len(text) for v in Counter(text).values()]
    return -sum(p * math.log2(p) for p in probs if p > 0)

In [6]:
def create_features(df, vectorizer=None, mode='train'):
    import numpy as np
    """Create features with TF-IDF and additional text statistics"""
    preprocessor = TextPreprocessor()
    
    df['clean_text'] = df['text'].apply(preprocessor.preprocess)

    df['text_length'] = df['text'].apply(len)
    df['word_count'] = df['text'].apply(lambda x: len(x.split()))
    df['avg_word_length'] = df['text'].apply(
        lambda x: numpy.mean([len(w) for w in x.split()]) if len(x.split()) > 0 else 0)

    df['non_latin'] = df['text'].apply(lambda x: len(re.findall(r'[^\x00-\x7F]', x))) 
    df['punct_count'] = df['text'].apply(lambda x: sum(1 for c in text if c in x))
    df['num_lines'] = df['text'].apply(lambda x: x.count('\n'))
    df['stop_count'] = df['text'].apply(lambda x: sum(1 for w in x if w in stopwords_set))
    df['long_words'] = df['text'].apply(lambda x:sum(1 for w in x if len(w) > 15))
    df['ent'] = df['text'].apply(lambda x: entropy(x))


    # Handle division by zero with small epsilon
    eps = 1e-10
    
    # Basic ratios (your current features)
    df['words_per_line'] = df['word_count'] / (df['num_lines'] + 1)
    df['punct_ratio'] = df['punct_count'] / (df['text_length'] + eps)
    df['stop_ratio'] = df['stop_count'] / (df['word_count'] + eps)
    df['long_words_ratio'] = df['long_words'] / (df['word_count'] + eps)
    df['non_latin_ratio'] = df['non_latin'] / (df['text_length'] + eps)
    
    # Advanced derived features
    df['entropy_per_word'] = df['ent'] / (df['word_count'] + eps)
    df['complexity_density'] = df['avg_word_length'] * df['words_per_line']
    df['linguistic_diversity'] = df['ent'] * df['stop_ratio']
    
    # Document structure features
    df['avg_line_length'] = df['text_length'] / (df['num_lines'] + 1)
    df['content_density'] = df['word_count'] / (df['text_length'] + eps)
    
    # Stylistic features
    df['formality_score'] = (df['long_words_ratio'] - df['stop_ratio']) * df['avg_word_length']
    df['readability_proxy'] = df['words_per_line'] / (df['avg_word_length'] + eps)
    
    # Handle any remaining NaN or inf values
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0)

    if mode == 'train':
        vectorizer = TfidfVectorizer(
            ngram_range=(1, 3),
            max_features=10000,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        X_tfidf = vectorizer.fit_transform(df['clean_text'])
        return X_tfidf, vectorizer, df
    else:
        if vectorizer is None:
            raise ValueError("Vectorizer must be provided for test mode")
        X_tfidf = vectorizer.transform(df['clean_text'])
        return X_tfidf, df


In [7]:
def train_models(X, y):
    """Train models with cross-validation and hyperparameter tuning"""
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)


    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


    models = {
        'LogisticRegression': LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            C=0.1,
            solver='saga',
            penalty='elasticnet',
            l1_ratio=0.5
        ),
        'RandomForest': RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            class_weight='balanced_subsample',
            random_state=42
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            random_state=42
        ),
        'SVM': CalibratedClassifierCV(
            SVC(
                kernel='rbf',
                C=1.0,
                gamma='scale',
                class_weight='balanced',
                probability=True
            ),
            cv=3
        )
    }

    trained_models = {}
    val_scores = {}


    for name, model in models.items():
        print(f"\nTraining {name}...")
        model.fit(X_train_res, y_train_res)
        trained_models[name] = model


        val_preds = model.predict(X_val)
        acc = accuracy_score(y_val, val_preds)
        f1 = f1_score(y_val, val_preds)
        val_scores[name] = {'accuracy': acc, 'f1': f1}

        print(f"{name} Validation Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")
        print(classification_report(y_val, val_preds))


    voting_clf = VotingClassifier(
        estimators=[(name, model) for name, model in trained_models.items()],
        voting='soft',
        n_jobs=-1
    )
    voting_clf.fit(X_train_res, y_train_res)
    trained_models['Ensemble'] = voting_clf


    val_preds = voting_clf.predict(X_val)
    acc = accuracy_score(y_val, val_preds)
    f1 = f1_score(y_val, val_preds)
    val_scores['Ensemble'] = {'accuracy': acc, 'f1': f1}
    print(f"\nEnsemble Validation Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")
    print(classification_report(y_val, val_preds))

    return trained_models, val_scores


In [8]:
#import h2o
#from h2o.automl import H2OAutoML
from autogluon.tabular import TabularPredictor, TabularDataset

def autoMLModel(X, X_test):
    #AutoML Solution
    label = 'real'
    #X = TabularDataset(X)
    #X_test = TabularDataset(X_test)
    predictor = TabularPredictor(label=label).fit(X)
    predictions = predictor.predict(X_test)
    probas = predictor.predict_proba(X_test)
    #print(len(X_test['folder'].str.extract(r'article_(\d+)').astype(int).to_numpy().flatten()))
    #print(len(predictions.values))
    result  = pd.DataFrame({
                    'id': X_test['folder'].str.extract(r'article_(\d+)').astype(int).to_numpy().flatten(),
                    'real_text_id': predictions.values,
                    'confidence': probas.max(axis=1)
                }).sort_values('id')
    best_confidence_idx = result.groupby('id')['confidence'].idxmax()

    # Extract those rows
    result = result.loc[best_confidence_idx, ['id', 'real_text_id']].reset_index(drop=True)

    return result

def autoMLModel_v2(X, X_test):
    #AutoML Solution
    label = 'real'
    X = TabularDataset(X)
    X_test = TabularDataset(X_test)
    predictor = TabularPredictor(label=label).fit(X)
    predictions = predictor.predict(X_test)
    probas = predictor.predict_proba(X_test)
    
    # Calculate balanced confidence (difference between top 2 probabilities)
    sorted_probas = numpy.sort(probas, axis=1)
    balanced_confidence = sorted_probas[:, -1] - sorted_probas[:, -2]  # highest - second highest
    
    result = pd.DataFrame({
        'id': X_test['folder'].str.extract(r'article_(\d+)').astype(int).to_numpy().flatten(),
        'real_text_id': predictions.values,
        'confidence': balanced_confidence
    }).sort_values('id')
    
    best_confidence_idx = result.groupby('id')['confidence'].idxmax()
    # Extract those rows
    result = result.loc[best_confidence_idx, ['id', 'real_text_id']].reset_index(drop=True)
    return result

# Specialized version for your specific data structure
def autoMLModel_specialized(X, X_test):
    """AutoML specialized for your text classification task"""
    import numpy as np
    #from autogluon.tabular import TabularDataset, TabularPredictor
    
    # 1. Prepare training data
    columns_to_drop = ['text', 'clean_text', 'folder', 'file_id']
    X_train = X.drop(columns=columns_to_drop, errors='ignore').copy()
    
    # Extract test IDs
    test_ids = X_test['folder'].str.extract(r'article_(\d+)').astype(int).values.flatten()
    X_test_features = X_test.drop(columns=columns_to_drop, errors='ignore').copy()
    
    # 2. Feature engineering - create additional features
    # Text complexity features
    X_train['words_per_line'] = X_train['word_count'] / (X_train['num_lines'] + 1)
    X_train['punct_ratio'] = X_train['punct_count'] / X_train['text_length']
    X_train['stop_ratio'] = X_train['stop_count'] / X_train['word_count']
    X_train['long_words_ratio'] = X_train['long_words'] / X_train['word_count']
    X_train['non_latin_ratio'] = X_train['non_latin'] / X_train['text_length']
    
    # Apply same feature engineering to test data
    X_test_features['words_per_line'] = X_test_features['word_count'] / (X_test_features['num_lines'] + 1)
    X_test_features['punct_ratio'] = X_test_features['punct_count'] / X_test_features['text_length']
    X_test_features['stop_ratio'] = X_test_features['stop_count'] / X_test_features['word_count']
    X_test_features['long_words_ratio'] = X_test_features['long_words'] / X_test_features['word_count']
    X_test_features['non_latin_ratio'] = X_test_features['non_latin'] / X_test_features['text_length']
    
    # Handle inf and nan values
    X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test_features = X_test_features.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # 3. Convert to TabularDataset
    train_data = TabularDataset(X_train)
    test_data = TabularDataset(X_test_features)
    
    # 4. Configure AutoML for imbalanced binary classification
    predictor = TabularPredictor(
        label='real',
        problem_type='binary',
        eval_metric='f1',
        path='./automl_specialized',
        verbosity=2
    )
    
    # 5. Train with focus on handling imbalanced data
    predictor.fit(
        train_data,
        time_limit=400,
        presets='good_quality',
        hyperparameters={
            'GBM': [
                {'num_boost_round': 300, 'learning_rate': 0.1, 'max_depth': 6},
                {'num_boost_round': 500, 'learning_rate': 0.05, 'max_depth': 8}
            ],
            'CAT': [
                {'iterations': 300, 'learning_rate': 0.1, 'depth': 6},
                {'iterations': 500, 'learning_rate': 0.05, 'depth': 8}
            ],
            'XGB': [
                {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 6},
                {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 8}
            ],
            'RF': [
                {'n_estimators': 200, 'max_depth': 10, 'class_weight': 'balanced'},
                {'n_estimators': 300, 'max_depth': 15, 'class_weight': 'balanced'}
            ]
        },
        num_bag_folds=3,
        auto_stack=True
    )
    
    # 6. Make predictions
    predictions = predictor.predict(test_data)
    probas = predictor.predict_proba(test_data)
    
    # 7. Calculate confidence
    if len(probas.columns) >= 2:
        prob_0 = probas.iloc[:, 0].values
        prob_1 = probas.iloc[:, 1].values
        balanced_confidence = np.abs(prob_1 - prob_0)
    else:
        balanced_confidence = probas.max(axis=1).values
    
    # 8. Create results and select best per article
    result = pd.DataFrame({
        'id': test_ids,
        'real_text_id': predictions.values,
        'confidence': balanced_confidence
    }).sort_values('id')
    
    best_confidence_idx = result.groupby('id')['confidence'].idxmax()
    result = result.loc[best_confidence_idx, ['id', 'real_text_id']].reset_index(drop=True)
    
    return result
    
# Debugging version to understand what's happening
def autoMLModel_debug(X, X_test):
    """Debug version to understand AutoML issues"""
    import numpy as np
    from autogluon.tabular import TabularDataset, TabularPredictor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    
    # 1. Prepare data exactly like your traditional ML
    columns_to_drop = ['text', 'clean_text', 'folder', 'file_id']
    X_train = X.drop(columns=columns_to_drop, errors='ignore').copy()
    # Extract test IDs
    test_ids = X_test['folder'].str.extract(r'article_(\d+)').astype(int).values.flatten()
    X_test_features = X_test.drop(columns=columns_to_drop, errors='ignore').copy()
    
    
    # Check class distribution
    print(f"Class distribution in training: {X_train['real'].value_counts()}")
    print(f"Class balance: {X_train['real'].mean():.3f}")

     # 2. Feature engineering - create additional features
    # Text complexity features
    X_train['words_per_line'] = X_train['word_count'] / (X_train['num_lines'] + 1)
    X_train['punct_ratio'] = X_train['punct_count'] / X_train['text_length']
    X_train['stop_ratio'] = X_train['stop_count'] / X_train['word_count']
    X_train['long_words_ratio'] = X_train['long_words'] / X_train['word_count']
    X_train['non_latin_ratio'] = X_train['non_latin'] / X_train['text_length']
    
    # Apply same feature engineering to test data
    X_test_features['words_per_line'] = X_test_features['word_count'] / (X_test_features['num_lines'] + 1)
    X_test_features['punct_ratio'] = X_test_features['punct_count'] / X_test_features['text_length']
    X_test_features['stop_ratio'] = X_test_features['stop_count'] / X_test_features['word_count']
    X_test_features['long_words_ratio'] = X_test_features['long_words'] / X_test_features['word_count']
    X_test_features['non_latin_ratio'] = X_test_features['non_latin'] / X_test_features['text_length']
    
    # 2. Split data to validate AutoML performance
    X_train_split, X_val_split, y_train, y_val = train_test_split(
        X_train.drop('real', axis=1), X_train['real'], 
        test_size=0.2, random_state=42, stratify=X_train['real']
    )
    
    # Recreate training data with target
    train_data_split = X_train_split.copy()
    train_data_split['real'] = y_train
    
    # 3. Simple AutoML configuration
    predictor = TabularPredictor(
        label='real',
        problem_type='binary',
        eval_metric='f1',
        path='./automl_debug',
        verbosity=3
    )
    
    # 4. Train with minimal configuration first
    predictor.fit(
        TabularDataset(train_data_split),
        time_limit=120,  # Just 2 minutes
        presets='medium_quality',
        hyperparameters='default',
        num_bag_folds=0,  # No bagging initially
        auto_stack=False  # No stacking initially
    )
    
    # 5. Test on validation set
    val_predictions = predictor.predict(TabularDataset(X_val_split))
    val_probas = predictor.predict_proba(TabularDataset(X_val_split))
    
    print("\nValidation Results:")
    print(f"Accuracy: {(val_predictions == y_val).mean():.4f}")
    print(f"F1 Score: {f1_score(y_val, val_predictions):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, val_predictions))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, val_predictions))
    
    # 6. Feature importance
    try:
        feature_importance = predictor.feature_importance(TabularDataset(train_data_split))
        print("\nTop 10 Feature Importances:")
        print(feature_importance.head(10))
    except:
        print("Could not get feature importance")
    
    # 7. Model leaderboard
    print("\nModel Leaderboard:")
    print(predictor.leaderboard())
    
    
    predictions = predictor.predict(TabularDataset(X_test_features))
    probas = predictor.predict_proba(TabularDataset(X_test_features))
    
    if len(probas.columns) >= 2:
        balanced_confidence = np.abs(probas.iloc[:, 1] - probas.iloc[:, 0]).values
    else:
        balanced_confidence = probas.max(axis=1).values
    
    result = pd.DataFrame({
        'id': test_ids,
        'real_text_id': predictions.values,
        'confidence': balanced_confidence
    }).sort_values('id')
    
    best_confidence_idx = result.groupby('id')['confidence'].idxmax()
    result = result.loc[best_confidence_idx, ['id', 'real_text_id']].reset_index(drop=True)
    
    return result

def autoMLModel_enhanced_v3(X, X_test):
    """Enhanced AutoML model with robust feature engineering and validation"""
    import numpy as np
    import pandas as pd
    from autogluon.tabular import TabularDataset, TabularPredictor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix, f1_score
    
    # 1. Prepare data
    columns_to_drop = ['text', 'clean_text', 'folder', 'file_id']
    X_train = X.drop(columns=columns_to_drop, errors='ignore').copy()
    
    # Extract test IDs
    test_ids = X_test['folder'].str.extract(r'article_(\d+)').astype(int).values.flatten()
    X_test_features = X_test.drop(columns=columns_to_drop, errors='ignore').copy()
    
    # Check class distribution
    print(f"Class distribution in training: {X_train['real'].value_counts()}")
    print(f"Class balance: {X_train['real'].mean():.3f}")

    
    # 3. Split data for validation
    X_train_split, X_val_split, y_train, y_val = train_test_split(
        X_train.drop('real', axis=1), X_train['real'], 
        test_size=0.2, random_state=42, stratify=X_train['real']
    )
    
    # Recreate training data with target
    train_data_split = X_train_split.copy()
    train_data_split['real'] = y_train
    
    # 4. Enhanced AutoML configuration
    predictor = TabularPredictor(
        label='real',
        problem_type='binary',
        eval_metric='f1',
        path='./automl_enhanced',
        verbosity=2
    )
    
    
    predictor.fit(
        TabularDataset(train_data_split),
        time_limit=300,  # 5 minutes
        presets='best_quality',
        num_bag_folds=5,  # Enable bagging for better performance
        auto_stack=True   # Enable stacking for ensemble
    )
    
    # 6. Validation evaluation
    val_predictions = predictor.predict(TabularDataset(X_val_split))
    val_probas = predictor.predict_proba(TabularDataset(X_val_split))
    
    print("\nValidation Results:")
    print(f"Accuracy: {(val_predictions == y_val).mean():.4f}")
    print(f"F1 Score: {f1_score(y_val, val_predictions):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, val_predictions))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_val, val_predictions))
    
    # 7. Feature importance analysis
    try:
        feature_importance = predictor.feature_importance(TabularDataset(train_data_split))
        print("\nTop 15 Feature Importances:")
        print(feature_importance.head(15))
    except Exception as e:
        print(f"Could not get feature importance: {e}")
    
    # 8. Model leaderboard
    print("\nModel Leaderboard:")
    leaderboard = predictor.leaderboard()
    print(leaderboard)
    
    # 9. Test predictions with ensemble
    predictions = predictor.predict(TabularDataset(X_test_features))
    probas = predictor.predict_proba(TabularDataset(X_test_features))
    
    # Calculate confidence scores
    if len(probas.columns) >= 2:
        # For binary classification, use the probability difference
        balanced_confidence = np.abs(probas.iloc[:, 1] - probas.iloc[:, 0]).values
    else:
        balanced_confidence = probas.max(axis=1).values
    
    # Create results DataFrame
    result = pd.DataFrame({
        'id': test_ids,
        'real_text_id': predictions.values,
        'confidence': balanced_confidence
    }).sort_values('id')

    # Handle multiple predictions per ID (keep highest confidence)
    best_confidence_idx = result.groupby('id')['confidence'].idxmax()
    #result_data = best_confidence_idx.copy()
    result = result.loc[best_confidence_idx, ['id', 'real_text_id']].reset_index(drop=True)
    
    # 10. Additional diagnostics
    print(f"\nFinal Results Summary:")
    print(f"Total test predictions: {len(result)}")
    print(f"Unique IDs: {result['id'].nunique()}")
    print(f"Predictions distribution: {result['real_text_id'].value_counts()}")
    #print(f"Average confidence: {result_data['confidence'].mean():.4f}")
    #print(f"Confidence std: {result_data['confidence'].std():.4f}")
    
    return result

In [9]:
def predict_test(models, val_scores, X_test, test_df):
    """Make predictions with fallback strategies"""

    proba_dfs = []
    for name, model in models.items():
        if hasattr(model, 'predict_proba'):
            try:
                proba = model.predict_proba(X_test)[:, 1]
                proba_dfs.append(pd.DataFrame({
                    'folder': test_df['folder'].values,
                    'file_id': test_df['file_id'].values,
                    f'proba_{name}': proba
                }))
            except Exception as e:
                print(f"⚠️ Error getting probabilities from {name}: {str(e)}")
                continue

    if not proba_dfs:
        raise ValueError("No probability data was generated from any model")


    proba_df = proba_dfs[0]
    for df in proba_dfs[1:]:
        proba_df = proba_df.merge(df, on=['folder', 'file_id'])


    try:
        proba_df['file_number'] = proba_df['file_id'].str.extract(r'file_(\d+)').astype(int)
        proba_df['article_id'] = proba_df['folder'].str.extract(r'article_(\d+)').astype(int)
    except Exception as e:
        print(f"⚠️ Error extracting file numbers or article IDs: {str(e)}")
        proba_df['file_number'] = proba_df['file_id'].apply(lambda x: int(x.split('_')[-1]))
        proba_df['article_id'] = proba_df.index


    best_model = max(val_scores.items(), key=lambda x: x[1]['f1'])[0]

    final_selection = []
    for article_id in proba_df['article_id'].unique():
        try:
            article_files = proba_df[proba_df['article_id'] == article_id].copy()

            if len(article_files) == 0:
                print(f"⚠️ No files found for article {article_id}")
                continue

            article_files['best_model_rank'] = article_files[f'proba_{best_model}'].rank(ascending=False)

            proba_cols = [col for col in article_files.columns if col.startswith('proba_')]
            article_files['ensemble_proba'] = article_files[proba_cols].mean(axis=1)
            article_files['ensemble_rank'] = article_files['ensemble_proba'].rank(ascending=False)

            selected = None
            try:
                best_model_choice = article_files[article_files['best_model_rank'] == 1].iloc[0]

                if best_model_choice[f'proba_{best_model}'] > 0.6:
                    selected = best_model_choice
                else:

                    ensemble_choice = article_files[article_files['ensemble_rank'] == 1].iloc[0]
                    selected = ensemble_choice
            except IndexError:

                selected = article_files.iloc[0]
                print(f"⚠️ Used fallback selection for article {article_id}")

            final_selection.append({
                'id': int(selected['article_id']),
                'real_text_id': int(selected['file_number']),
                'confidence': max(selected[f'proba_{best_model}'], selected.get('ensemble_proba', 0))
            })

        except Exception as e:
            print(f"⚠️ Error processing article {article_id}: {str(e)}")
            continue

    if not final_selection:
        raise ValueError("No articles were processed successfully")

    submission_df = pd.DataFrame(final_selection)
    return submission_df.sort_values('id')


In [10]:
if __name__ == "__main__":
    data_dir = "/kaggle/input/fake-or-real-the-impostor-hunt/data/train"
    train_csv_path = "/kaggle/input/fake-or-real-the-impostor-hunt/data/train.csv"
    test_path = "/kaggle/input/fake-or-real-the-impostor-hunt/data/test"

    print("Loading training data...")
    df = load_data(data_dir, train_csv_path)

    print("Loading test data...")
    test_data = []
    for folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, folder)
        if os.path.isdir(folder_path):
            for file_id in ["1", "2"]:
                file_path = os.path.join(folder_path, f"file_{file_id}.txt")
                try:
                    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
                        text = f.read().strip()
                    if text:
                        test_data.append({'folder': folder, 'file_id': f'file_{file_id}', 'text': text})
                except (FileNotFoundError, UnicodeDecodeError) as e:
                    print(f"⚠️ Error loading test file {file_path}: {str(e)}")
                    continue
    test_df = pd.DataFrame(test_data)


    print("Creating features...")
    X, vectorizer, df = create_features(df, mode='train')
    y = df['real'].astype(int)


    if test_df.empty:
        print("⚠️ Warning: No test data was loaded!")
        submission_df = pd.DataFrame(columns=['id', 'real_text_id'])
    else:
        X_test, test_df = create_features(test_df, vectorizer=vectorizer, mode='test')

        if MODEL == 'AutoML':
            print("\nTraining AutoML model...")
            submission_df = autoMLModel_enhanced_v3(df, test_df)
        else:
            print("\nTraining models...")
            trained_models, val_scores = train_models(X, y)
            
            print("\nMaking predictions...")
            submission_df = predict_test(trained_models, val_scores, X_test, test_df)

    submission_df[['id', 'real_text_id']].to_csv("submission.csv", index=False)
    print("\n✅ Enhanced submission created: submission.csv")


Loading training data...
Loading test data...
Creating features...


Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Dynamic stacking is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
Detecting stacked overfitting by sub-fitting AutoGluon on the input data. That is, copies of AutoGluon will be sub-fit on subset(s) of the data. Then, the holdout validation data is used to detect stacked overfitting.
Sub-fit(s) time limit is: 300 seconds.
Starting holdout-based sub-fit for dynamic stacking. Context path is: ./automl_enhanced/ds_sub_fit/sub_fit_ho.



Training AutoML model...
Class distribution in training: real
1    95
0    93
Name: count, dtype: int64
Class balance: 0.505


Beginning AutoGluon training ... Time limit = 75s
AutoGluon will save models to "./automl_enhanced/ds_sub_fit/sub_fit_ho"
=================== System Info ===================
AutoGluon Version:  1.0.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.91 GB / 31.35 GB (95.4%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Train Data Rows:    133
Train Data Columns: 21
Label Column:       real
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 = 1, class 0 = 0
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    30633.02 MB
	Train Data (Original)  Memory Usage: 0.02 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the fe


Validation Results:
Accuracy: 0.8421
F1 Score: 0.8500

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.79      0.83        19
           1       0.81      0.89      0.85        19

    accuracy                           0.84        38
   macro avg       0.85      0.84      0.84        38
weighted avg       0.85      0.84      0.84        38


Confusion Matrix:
[[15  4]
 [ 2 17]]


/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")
/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you t


Top 15 Feature Importances:
                      importance    stddev   p_value  n  p99_high   p99_low
stop_ratio              0.023638  0.007450  0.001042  5  0.038977  0.008300
complexity_density      0.022863  0.016211  0.017198  5  0.056242 -0.010516
num_lines               0.021053  0.011796  0.008126  5  0.045341 -0.003235
linguistic_diversity    0.016366  0.012396  0.020938  5  0.041889 -0.009157
punct_ratio             0.016009  0.012942  0.025267  5  0.042656 -0.010638
non_latin_ratio         0.015084  0.017158  0.060372  5  0.050413 -0.020244
readability_proxy       0.014829  0.014239  0.040181  5  0.044148 -0.014489
avg_line_length         0.014693  0.012448  0.028808  5  0.040324 -0.010937
words_per_line          0.012646  0.009941  0.023325  5  0.033113 -0.007822
entropy_per_word        0.012590  0.006492  0.006145  5  0.025957 -0.000778
punct_count             0.009971  0.005831  0.009360  5  0.021977 -0.002035
formality_score         0.007522  0.007599  0.045645  5  0.

/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")
/usr/local/lib/python3.11/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you t


Final Results Summary:
Total test predictions: 1068
Unique IDs: 1068
Predictions distribution: real_text_id
0    545
1    523
Name: count, dtype: int64

✅ Enhanced submission created: submission.csv


In [11]:
#df.head()

In [12]:
#test_df.head()

In [13]:
submission_df

,id,real_text_id
0,0,1
1,1,1
2,2,0
3,3,1
4,4,1
...,...,...
1063,1063,1
1064,1064,0
1065,1065,0
1066,1066,0
